Import libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#%matplotlib inline
import pandas as pd
import math
import numpy as np
from datetime import datetime, timedelta
from scipy import stats
import warnings
from IPython.display import display
import ipywidgets as widgets
pd.options.mode.chained_assignment = None 
from datetime import timezone
import datetime
from suntime import Sun
import pytz
from numpy.linalg import norm
from sklearn.metrics.pairwise import cosine_similarity

Read CSV

In [3]:
dataset = pd.read_csv("10sec-original.csv")

dataset['datetime'] = pd.to_datetime(dataset['datetime'], format='mixed')
lat = 18.47
lon = -66.12
tz = pytz.timezone('America/Puerto_Rico')

In [4]:
dataset

,tagID,datetime,event
0,1621,2017-06-21 08:00:10.850,entering
1,1621,2017-06-21 08:10:33.850,exiting
2,1621,2017-06-21 08:24:48.250,entering
3,1621,2017-06-21 08:51:23.450,entering
4,1621,2017-06-21 08:55:48.300,exiting
...,...,...,...
1855,661,2017-06-22 17:09:08.250,entering
1856,2258,2017-06-22 11:49:10.700,entering
1857,2258,2017-06-22 12:07:42.750,exiting
1858,2258,2017-06-22 16:14:06.350,entering


Data cleaning functions

Separate entry/exit events and convert into flights, takes entry/exit dataset
for a single bee and returns a table.

In [5]:

def classifyLoc(dataset):

    bee = dataset
    
    cur = bee['event']
    nxt = cur.shift(-1)
    cur_dt = bee['datetime']
    nxt_dt = cur_dt.shift(-1)
    #selects all except last
    valid = nxt.notna()
    location = np.select(
        [
            (cur == "entering") & (nxt == "exiting"),
            (cur == "entering") & (nxt == "exiting-entering"),
            
            (cur == "entering-exiting") & (nxt == "entering"),
            (cur == "exiting") & (nxt == "entering"),
            (cur == "exiting") & (nxt == "entering-exiting"),
            
            (cur == "entering") & (nxt == "entering"),
            (cur == "outside") & (nxt == "outside"),
        ],
        ["Inside", "Inside", "Outside", "Outside", "Outside", "Unknown-Inside", "Outside-Unknown"],
        default="Unknown",
    )
            
    t0 = cur_dt.iloc[0]
    age = (cur_dt - t0).dt.days

    #new dataset with inside/outside and time start/end
    BeeTravel = pd.DataFrame(
        {
            "location": location[valid],
            "start": cur_dt[valid],
            "end": nxt_dt[valid],
            "age": age[valid]
        }
    ).reset_index(drop=True)
    
    #classify short outside/insides as part of another group
    BeeTravel['activity'] = "undefined"
    BeeTravel.loc[BeeTravel["location"] == "Inside", "activity"] = "inside"
    duration = (BeeTravel["end"] - BeeTravel["start"]).dt.total_seconds()
    outside = BeeTravel["location"].isin(["Outside"])
   
    #Classify outside flights depending on length
    BeeTravel.loc[outside & (duration < 300), "activity"] = "outside-short"
    BeeTravel.loc[outside & (duration >= 300) & (duration  < 7200), "activity"] = "outside-foraging"
    BeeTravel.loc[outside & (duration >= 7200), "activity"] = "outside-long"
    BeeTravel['tagID'] = dataset['tagID'].iloc[0]

    return BeeTravel



Utilizes the previous function on every individual bee to have a full table.

In [6]:
def cleanData(data):

    activity = []
    
    allActivity = (
        data.sort_values(["tagID","datetime"])
        .groupby("tagID", group_keys = False)[data.columns]
        .apply(classifyLoc)
        .reset_index(drop=True)
    )
    
    
    df = (allActivity[allActivity['location'].isin(["Outside","Outside-Unknown"])]
        [["tagID","start","end"]]
        .rename(columns={'start':'tripStart','end':'tripEnd'})
        .copy()
    )
     
     
    df['duration'] = df['tripEnd'] - df['tripStart']

    return allActivity, df

In [7]:
def fixHourMin(duration):
    hour = int(duration.total_seconds() // 3600)
    mins = int(duration.total_seconds() // 60 % 60)
    if mins < 10:
        mins = f'0{mins}'
    return f'{hour}:{mins}'

Creates a statistical summary of an entire hive based on two tables, all activity and all flights.
Returns a table containing the statistical summary.

Does so by iterating over every beeID separately.

In [8]:
def summaryData(activity,flights):
    
    
    seen = activity.groupby("tagID").agg(
        firstSeen = ('start','min'),
        lastSeen = ('end','max')
    )
    
    seen['firstSeen'] = seen['firstSeen'].dt.floor("s")
    seen['lastSeen'] = seen['lastSeen'].dt.floor("s")
    
    indFlight = flights.groupby("tagID").agg(
        firstFlight = ('tripStart','min'),
        lastFlight = ('tripStart','max'),
        avgTrip = ('duration','mean'),
        medianTrip = ('duration','median'),
        numTrips = ('duration','size')
    )
    
    indFlight['firstFlight'] = indFlight['firstFlight'].dt.floor("s").apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    indFlight['lastFlight'] = indFlight['lastFlight'].dt.floor("s").apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    indFlight['avgTrip'] = indFlight['avgTrip'].map(fixHourMin)
    indFlight['medianTrip'] = indFlight['medianTrip'].map(fixHourMin)
    
    tpd = flights.groupby("tagID").size().rename("tripsPerDay")
    
    full = seen.join(indFlight, how="left").join(tpd, how="left")
    full["tripsPerDay"].fillna(0, inplace = True)
      
    divisor = (full['lastSeen'].dt.date - full['firstSeen'].dt.date).apply(lambda x: x.days) + 1
    full['tripsPerDay'] = full["tripsPerDay"]/divisor
    full['tripsPerDay'] = full["tripsPerDay"].apply(lambda x: x if x > 0 else None)
    
    full['firstSeen'] = full['firstSeen'].apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    full['lastSeen'] = full['lastSeen'].apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    longest = flights.groupby('tagID')['duration'].idxmax()
    shortest = flights.groupby('tagID')['duration'].idxmin()
    
    long_df = flights.loc[longest, ["tagID", "tripStart", "duration"]].set_index("tagID")
    short_df = flights.loc[shortest, ["tagID", "tripStart", "duration"]].set_index("tagID")
    
    long_str = (
        long_df["tripStart"].dt.floor("s").astype(str)
        + " ~ "
        + long_df["duration"].map(fixHourMin)
    ).rename("longestFlight")
    
    short_str = (
        short_df["tripStart"].dt.floor("s").astype(str)
        + " ~ "
        + short_df["duration"].map(fixHourMin)
    ).rename("shortestFlight")
    
    full = full.join(long_str, how="left").join(short_str, how="left")
    
    beeSummary = full.rename(columns={"firstSeen":"First Seen", "lastSeen":"Last Seen","firstFlight":"First Flight",
                            "lastFlight":"Last Flight", "longestFlight":"Longest Flight","shortestFlight":"Shortest Flight",
                            "avgTrip":"Average Flight Duration", "numTrips":"# of Flights","tripsPerDay":"Average Flights per Day", "medianTrip":"Median Flight Duration"})
    beeSummary = beeSummary.reset_index()

    return beeSummary

Total summary table creation with values of all hive.

In [9]:
def makeTotalSum(flights):
    #general overview
    totalFlights = len(flights)
    flights['day'] = flights['tripStart'].apply(lambda x: x.date())
    totalPerDay = flights['day'].value_counts().mean()
    totalLength = flights['duration'].mean()
    totalLength = totalLength.total_seconds()/60

    #longest and shortest flights
    longest = flights['duration'].idxmax()
    shortest = flights['duration'].idxmin()

    longestDate = flights.loc[longest]['tripStart'].strftime('%Y-%m-%d ~ %H:%M:%S')
    longestID = flights.loc[longest]['tagID']
    longestLen = flights.loc[longest]['duration'].total_seconds()/60

    shortestDate = flights.loc[shortest]['tripStart'].strftime('%Y-%m-%d ~ %H:%M:%S')
    shortestID = flights.loc[shortest]['tagID']
    shortestLen = flights.loc[shortest]['duration'].total_seconds()/60

    #create tables
    #general info
    totalDict = {"Total Flights":totalFlights, "Avg Flights per Day":totalPerDay, "Avg Length per Day (minutes)":totalLength}
    totalSum = pd.DataFrame.from_dict(totalDict, orient='index')
    totalSum = totalSum.rename(columns={0: "Overall Statistics"})

    #info on longest/shortest trip
    tripDict = {"Classification":["Longest Flight", "Shortest Flight"],"Date":[longestDate, shortestDate], "BeeID":[longestID, shortestID], "Length (minutes)":[longestLen, shortestLen]}
    tripSum = pd.DataFrame.from_dict(tripDict)
    
    return totalSum, tripSum

Obtain sunrise and sunset table for actograph based on previously defined timezone and required days for table.

In [10]:
def getSRSS(dataset, tz, lat, lon):
    dates = dataset['start'].apply(lambda x: x.date()).sort_values()
    dates = dates.unique()
    sunrise = []
    sunset = []
    sun = Sun(lat, lon)
    for d in dates:
        dT = datetime.datetime(d.year, d.month, d.day)
        sunrise.append(sun.get_sunrise_time(dT).astimezone(tz).time())
        sunset.append(sun.get_sunset_time(dT).astimezone(tz).time())

    srss = pd.DataFrame.from_dict({"date":dates,"sunrise":sunrise,"sunset":sunset})
    return srss


Split overnight flights at midnight for plotting reasons

missing = True and timeam/timepm for when a dataset has missing values depending
on dataset requirement.

In [11]:
def separateFlights(flights,missing=False,timeam="08:00:00",timepm="18:00:00"):
    newdata = []
    
    for i in range(len(flights)):
        row = flights.iloc[i].copy()
        row['day'] = row['tripStart'].to_pydatetime().day
        day = row['day']
        month0 = row['tripStart'].to_pydatetime().month
        month1 = row['tripEnd'].to_pydatetime().month
        
        if row['tripStart'].to_pydatetime().day != row['tripEnd'].to_pydatetime().day:
            #first day
            newentry = row.copy()
            newentry['tripEnd'] = row['tripEnd'].replace(hour=23,minute=59,second=59,day=1,month=1)
            newentry['tripStart'] = row['tripStart'].replace(day=1,month=1)
            newentry['date'] = fixDate(row['tripStart'])
            newdata.append(newentry)
            #second day
            newday = row['tripEnd'].to_pydatetime().day
            newentry = row.copy()
            newentry['tripStart'] = row['tripStart'].replace(day=1,hour=0,minute=0,second=0,month=1)
            newentry['tripEnd'] = row['tripEnd'].replace(day=1,month=1)
            newentry['day'] = newday
            newentry['date'] = fixDate(row['tripEnd'])
            newdata.append(newentry)
        else:
            newentry = row.copy()
            newentry['tripStart'] = row['tripStart'].replace(day=1,month=1)
            newentry['tripEnd'] = row['tripEnd'].replace(day=1,month=1)
            newentry['date'] = fixDate(row['tripStart'])
            newdata.append(newentry)

    if missing:
        cutoffday = str(newentry['tripStart'].date())
        #Extra for missing dataset
        for i in range(len(newdata)):
            cutoffpm = datetime.strptime(cutoffday + " " + timepm,"%Y-%m-%d %H:%M:%S")
            cutoffam = datetime.strptime(cutoffday + " " + timeam,"%Y-%m-%d %H:%M:%S")
            if newdata[i]['tripEnd'] >= cutoffpm:
                newrow = newdata[i].copy()
                newdata[i]['tripEnd'] = cutoffpm
                newdata[i]['Data'] = 'Available'
                newrow['tripStart'] = cutoffpm
                newrow['Data'] = 'Missing'
                newdata.append(newrow)
            elif newdata[i]['tripStart'] <= cutoffam:
                newrow = newdata[i].copy()
                newdata[i]['tripStart'] = cutoffam
                newdata[i]['Data'] = 'Available'
                newrow['tripEnd'] = cutoffam
                newrow['Data'] = 'Missing'
                newdata.append(newrow)
            else:
                newdata[i]['Data'] = "Available"
    else:
        for i in range(len(newdata)):
            newdata[i]['Data'] = 'Available'
    
    
    flights_mod = pd.concat(newdata, axis = 1)
    flights_mod = flights_mod.transpose().reset_index()
    flights_mod = flights_mod.drop(["index"],axis=1)
    return flights_mod


Apply shapes to time graph based on sunrise/sunset dataset.

Create time plot for bee flights utilizing flights dataset.

In [12]:
#function to change hour column into hour properly

def fixTime(hr):
    if hr < 10:
        return "0" + str(hr) + ":00"
    else:
        return str(hr) + ":00"
        
#function to change date column appropriately

def fixDate(x):
    if x.day > 9 and x.month > 9:
        return f'{x.month}/{x.day}'
    elif x.day > 9 and x.month < 10:
        return f'0{x.month}/{x.day}'
    elif x.day < 10 and x.month > 9:
        return f'{x.month}/0{x.day}'
    else:
        return f'0{x.month}/0{x.day}'

Prepare all new datasets based on csv

In [13]:
allActivity, flights = cleanData(dataset)
beeSummary = summaryData(allActivity,flights)
dt = getSRSS(allActivity, tz, lat, lon)
dt['date'] = dt['date'].apply(lambda x: fixDate(x))
Summary0, Summary1 = makeTotalSum(flights)


flights['date'] = flights['tripStart'].apply(lambda x: fixDate(x))
flights['hour'] = flights['tripStart'].apply(lambda x: x.hour)
flights['day'] = flights['tripStart'].apply(lambda x: x.day)
flights_mod = separateFlights(flights)

In [14]:
flights

,tagID,tripStart,tripEnd,duration,day,date,hour
1,197,2017-06-21 08:11:24.300,2017-06-21 08:28:03.650,0 days 00:16:39.350000,21,06/21,8
3,197,2017-06-21 08:37:01.650,2017-06-21 08:58:19.050,0 days 00:21:17.400000,21,06/21,8
7,197,2017-06-21 11:36:20.200,2017-06-21 13:10:58.100,0 days 01:34:37.900000,21,06/21,11
20,197,2017-06-22 16:13:54.750,2017-06-24 09:07:51.850,1 days 16:53:57.100000,22,06/22,16
22,197,2017-06-24 10:24:33.900,2017-06-24 15:44:37.050,0 days 05:20:03.150000,24,06/24,10
...,...,...,...,...,...,...,...
1813,2487,2017-06-22 08:23:36.650,2017-06-22 09:18:19.850,0 days 00:54:43.200000,22,06/22,8
1818,2487,2017-06-22 13:10:09.450,2017-06-22 15:16:52.900,0 days 02:06:43.450000,22,06/22,13
1821,2487,2017-06-23 09:08:55.000,2017-06-23 09:38:04.300,0 days 00:29:09.300000,23,06/23,9
1825,2487,2017-06-24 08:36:48.600,2017-06-24 09:49:40.550,0 days 01:12:51.950000,24,06/24,8


Divide bees in day-focus/afternoon-focus

In [15]:
def divideBees(dataset):
    hour = dataset['hour'].unique()
    hour.sort()
    split = np.where(hour == 12)[0][0]
    final = {"morning":[],"afternoon":[],"even":[]}
    vectors = {}
    maxcount = dataset[['tagID','hour']].value_counts().max()
    divisor = 10 ** math.ceil(math.log10(maxcount))
    #print(split)
    for i in dataset['tagID'].unique():
        bee = dataset[dataset['tagID'] == i]
        data, bins = np.histogram(bee['hour'], bins=hour,density=True)
        vector = bee['hour'].value_counts().reindex(hour, fill_value=0)
        vector = vector.sort_index()
        vector = vector/divisor
        #print(i)
        #print(data)
        morning = sum(data[0:split])
        afternoon = sum(data[split:])
        if morning > 0.7:
            final['morning'].append(i)
        elif afternoon > 0.7:
            final['afternoon'].append(i)
        else:
            final['even'].append(i)

        vectors[str(i)] = vector.tolist()

    return final,vectors
        

division, vectors = divideBees(flights)

Cosine similarity for collapsed vectors

In [16]:
threshold1 = 0.8
def findSimilar(vectors, tagID):
    similars = []
    current = vectors[str(tagID)]
    for key,value in vectors.items():
        A = value
        B = current
        cosine = np.dot(A, B) / (norm(A) * norm(B))
        if cosine > threshold1:
            similars.append(key)
    return(similars)

findSimilar(vectors,1621)

['197', '508', '1237', '1361', '1621', '1780', '2487']

In [17]:
#matrix for every day where 1 = activity, 0 = no activity (matrixes_simp)
#matrix where each value is % of hour spent outside (matrixes)


def createMatrix(df):
    df = df.copy()

    df.sort_values(by='date')
    df['day'] = df['date'].astype('category').cat.codes
    df['tripStart'] = pd.to_datetime(df['tripStart'])
    df['tripEnd']   = pd.to_datetime(df['tripEnd'])
    df['start_hour'] = df['tripStart'].dt.hour
    df['end_hour'] = df['tripEnd'].dt.hour

    n_days = df['day'].nunique()

    result = {}
    result_simple = {}
    result_count = {}

    for tag, g in df.groupby('tagID'):
        matrix = np.zeros((n_days, 24))
        matrix_simple = np.zeros((n_days, 24))
        matrix_count = np.zeros((n_days, 24))

        for index, row in g.iterrows():
            matrix_simple[row.day, row.start_hour:row.end_hour+1] = 1
            matrix_count[row.day, row.start_hour] += 1
            start_h = row.tripStart.hour
            end_h   = row.tripEnd.hour

            for h in range(start_h, end_h + 1):
                hour_start = row.tripStart.floor('D') + pd.Timedelta(hours=h)
                hour_end   = hour_start + pd.Timedelta(hours=1)

                overlap = (
                    min(row.tripEnd, hour_end)
                    - max(row.tripStart, hour_start)
                ).total_seconds()

                if overlap > 0:
                    matrix[row.day, h] += overlap / 3600

                    
        result[str(tag)] = matrix
        result_simple[str(tag)] = matrix_simple
        result_count[str(tag)] = matrix_count

    return result, result_simple, result_count
            
matrixes, matrixes_simp, matrixes_count = createMatrix(flights)

bin size experimentation

In [18]:
#flatten matrix and find cosine similarity of flattened matrix (vector)
threshold2 = 0.6
flattened = {}
for k in matrixes:
    flat = matrixes[k].copy()
    flat = flat.flatten()
    flattened[k] = flat

def findSimilarMatrix(vectors, tagID):
    similars = []
    current = vectors[str(tagID)]
    for key,value in vectors.items():
        A = value
        B = current
        cosine = np.dot(A, B) / (norm(A) * norm(B))
        if cosine > threshold2:
            similars.append(key)
    return(similars)
    
findSimilarMatrix(flattened,1621)

['1361', '1621']

In [19]:
#find cosine similarity of each day's vector, and then get score from how many days are similar
def findSimilarDay(matrixes, tagID, threshold):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        A = value
        B = current
        #count of all cosine similarities
        Acount = []
        for day in range(len(A)):
            Avec = A[day]
            Bvec = B[day]
            if norm(Avec) == 0 and norm(Bvec) == 0:
                Acount.append(1)
                continue
            elif (norm(Avec) * norm(Bvec)) == 0:
                Acount.append(0)
                continue
            cosine = np.dot(Avec, Bvec) / (norm(Avec) * norm(Bvec))
            if cosine >= threshold:
                Acount.append(1)
            else:
                Acount.append(0)
        if sum(Acount)/len(Acount) >= threshold:
            similars.append(key)
    return(similars)

findSimilarDay(matrixes_simp,1621,0.7)

['1361', '1621']

In [20]:
#find cosine similarity of each day's vector, and then get score from how many days are similar
def findSimilarDay(matrixes, tagID, threshold):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        A = value
        B = current
        #count of all cosine similarities
        Acount = []
        for day in range(len(A)):
            Avec = A[day]
            Bvec = B[day]
            if norm(Avec) == 0 and norm(Bvec) == 0:
                Acount.append(1)
                continue
            elif (norm(Avec) * norm(Bvec)) == 0:
                Acount.append(0)
                continue
            cosine = np.dot(Avec, Bvec) / (norm(Avec) * norm(Bvec))
            if cosine >= threshold:
                Acount.append(1)
            else:
                Acount.append(0)
        if sum(Acount)/len(Acount) >= threshold:
            similars.append(key)
    return(similars)

findSimilarDay(matrixes,1621,0.7)

['1361', '1621']

In [21]:
def similarFroNorm(matrixes,tagID):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        dif = (current - value)
        n = norm(dif, ord="fro")
        print(key)
        print(n/norm(current,ord="fro"))

print("TIME MATRIX")
similarFroNorm(matrixes,1621)
print("ACTIVITY MATRIX")
similarFroNorm(matrixes_simp,1621)

TIME MATRIX
197
1.14135901106947
508
1.1492148004768075
515
1.248321064240831
516
0.9982112363355458
517
1.047056029599802
638
0.9301910133096223
661
1.0851031801410649
677
0.9683506832625238
930
1.010970740392928
1230
0.9169514980541443
1237
1.22627283149459
1361
1.0884964776682968
1404
1.0069826538839097
1407
1.036153225578213
1415
1.002021507016564
1607
1.057124284736037
1609
1.053216769170626
1621
0.0
1651
0.9999086837955651
1652
1.0540116520124556
1698
0.997143822781802
1722
1.1773292391693118
1780
1.0859047168319231
1786
1.066524506286424
1797
1.1319287257444481
1900
0.9994981680195144
1979
1.0000528028395235
2258
1.1529928906592364
2487
1.0668308421184136
ACTIVITY MATRIX
197
1.0339078862458586
508
1.0170952554312156
515
1.1141720290623112
516
1.0504514628777806
517
1.1596670152276025
638
0.9284766908852594
661
1.0667385033281394
677
0.909717652294684
930
0.909717652294684
1230
0.9284766908852594
1237
1.1596670152276025
1361
0.8709883407113855
1404
0.9649012813540154
1407
1.09858

In [22]:
def similarSpecNorm(matrixes,tagID):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        dif = current - value
        n = norm(dif, ord=2)
        print(key)
        print(n/norm(current, ord=2))

similarSpecNorm(matrixes,1621)

197
1.1375623059542816
508
1.0429719175906194
515
1.145845909957538
516
0.9956182975762087
517
0.9426270116094005
638
0.865573650665705
661
0.981428088808476
677
0.8952873994738805
930
0.9597841349539907
1230
0.9156596695102815
1237
1.1830430571980541
1361
1.1123697010527813
1404
0.9865892223291918
1407
0.9659577068493876
1415
0.9170145872257528
1607
0.9598800615876958
1609
0.9558309913699674
1621
0.0
1651
0.9971143132466597
1652
0.9579312690179217
1698
0.9937932977744129
1722
1.0445971909638023
1780
1.0361951846845432
1786
0.8931897020749682
1797
1.0559209499496267
1900
0.9973748284713884
1979
0.9983537593170445
2258
1.0339659987579037
2487
0.8999157857033094


In [23]:
flights['duration'].mean()

Timedelta('0 days 01:01:08.235208711')

In [24]:
flights['duration'].median()

Timedelta('0 days 00:02:58.750000')

In [25]:
flights['duration'].std()

Timedelta('0 days 03:57:49.516563168')

In [26]:
allActivity['duration'] = (allActivity['end'] - allActivity['start']).apply(lambda x: x.total_seconds())
allActivity['day'] = allActivity['start'].dt.day
allActivity['daydiff'] = (allActivity['end'].dt.day - allActivity['start'].dt.day)
dormant = allActivity[(allActivity['location'] == "Inside") & (allActivity['daydiff'] == 0)]
flights = allActivity[(allActivity['location'] == "Outside") & (allActivity['daydiff'] == 0)]
flights['date'] = flights['start'].apply(lambda x: fixDate(x))

In [27]:
flights

,location,start,end,age,activity,tagID,duration,day,daydiff,date
1,Outside,2017-06-21 08:11:24.300,2017-06-21 08:28:03.650,0,outside-foraging,197,999.35,21,0,06/21
3,Outside,2017-06-21 08:37:01.650,2017-06-21 08:58:19.050,0,outside-foraging,197,1277.40,21,0,06/21
7,Outside,2017-06-21 11:36:20.200,2017-06-21 13:10:58.100,0,outside-foraging,197,5677.90,21,0,06/21
22,Outside,2017-06-24 10:24:33.900,2017-06-24 15:44:37.050,3,outside-long,197,19203.15,24,0,06/24
23,Outside,2017-06-21 10:20:14.450,2017-06-21 12:15:25.800,0,outside-foraging,508,6911.35,21,0,06/21
...,...,...,...,...,...,...,...,...,...,...
1813,Outside,2017-06-22 08:23:36.650,2017-06-22 09:18:19.850,0,outside-foraging,2487,3283.20,22,0,06/22
1818,Outside,2017-06-22 13:10:09.450,2017-06-22 15:16:52.900,1,outside-long,2487,7603.45,22,0,06/22
1821,Outside,2017-06-23 09:08:55.000,2017-06-23 09:38:04.300,1,outside-foraging,2487,1749.30,23,0,06/23
1825,Outside,2017-06-24 08:36:48.600,2017-06-24 09:49:40.550,2,outside-foraging,2487,4371.95,24,0,06/24


In [28]:
flights_count = flights.copy()
flights_count['half_hour'] = flights_count['start'].apply(lambda x: x.hour + round(x.minute/30)/2)
flights_count['20minute']= flights_count['start'].apply(lambda x: x.hour + math.floor((x.minute/20) * 3)/3)
flights_count['10minute']= flights_count['start'].apply(lambda x: x.hour + math.floor((x.minute/10) * 6)/6)
flights_count['5minute'] = flights_count['start'].apply(lambda x: x.hour + math.floor((x.minute/20) * 12)/12)

## FREQUENCY (D1)

Without filling empty bins

In [29]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import wasserstein_distance

In [30]:
flights_nofill = flights.groupby(['tagID','date']).size().unstack()
x = flights_nofill.apply(lambda x: [x.dropna().mean(), x.dropna().std(), stats.variation(x.dropna()), stats.skew(x.dropna())] if len(x.dropna()) > 3 and  x.dropna().std() != 0 else pd.NA, axis=1)
x = x.dropna()

In [31]:
x

tagID
508     [2.0, 1.5491933384829668, 0.7071067811865476, ...
515     [1.75, 0.9574271077563381, 0.47380354147934284...
516     [1.125, 0.3535533905932738, 0.2939723678960656...
517     [5.142857142857143, 2.4784787961282104, 0.4461...
638     [3.3333333333333335, 0.816496580927726, 0.2236...
1237    [4.0, 2.0, 0.4629100498862757, 1.2151388809514...
1361    [2.7142857142857144, 1.7043362064926932, 0.581...
1407                  [6.0, 3.4641016151377544, 0.5, 0.0]
1607    [15.375, 9.28804915700047, 0.5650845160489477,...
1609    [4.142857142857143, 2.7945525240230875, 0.6245...
1621    [4.0, 2.3664319132398464, 0.5400617248673217, ...
1652    [1.5714285714285714, 0.7867957924694432, 0.463...
1698    [4.5, 3.7859388972001824, 0.7286042804780001, ...
1722    [2.1666666666666665, 1.1690451944500122, 0.492...
1780    [4.0, 4.96655480858378, 1.14953406710222, 1.85...
1797    [3.375, 2.38671920665766, 0.6615026333145093, ...
1900    [2.75, 2.8722813232690143, 0.9045340337332909,...
2487    

In [32]:
array = np.array(x.tolist())
squareform(pdist(array))

array([[ 0.        ,  1.14675545,  1.75910411,  3.71572018,  2.5906119 ,
         2.0742443 ,  1.19563282,  4.65937409, 15.53535671,  2.61485742,
         2.5876211 ,  1.06372699,  3.38556276,  1.03778608,  4.0081583 ,
         1.93942889,  1.5726816 ,  2.04114037],
       [ 1.14675545,  0.        ,  1.98397065,  3.80561249,  1.96026593,
         2.58273297,  1.22456957,  4.95883347, 15.98448695,  3.02189798,
         2.70103882,  0.44208457,  3.980371  ,  0.46773764,  4.84115054,
         2.17811592,  2.27596757,  1.11121101],
       [ 1.75910411,  1.98397065,  0.        ,  5.23098226,  3.67022877,
         3.48037629,  2.76441581,  6.21501728, 16.99884455,  4.24353448,
         4.18577616,  1.54852053,  5.00774629,  2.22681651,  5.51821312,
         3.61545756,  3.28661143,  3.04300827],
       [ 3.71572018,  3.80561249,  5.23098226,  0.        ,  2.48628096,
         1.97054112,  2.67304535,  1.34520684, 12.29225321,  1.39467446,
         1.19500192,  4.12340361,  1.95631187,  3.349

In [33]:
x = flights_nofill.apply(lambda x: [v for v in x if pd.notna(v)],axis=1)
x

tagID
197                                        [3.0, 1.0]
508                    [1.0, 1.0, 2.0, 2.0, 5.0, 1.0]
515                              [3.0, 2.0, 1.0, 1.0]
516          [1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0]
517               [4.0, 8.0, 7.0, 2.0, 2.0, 7.0, 6.0]
638                    [4.0, 4.0, 4.0, 3.0, 2.0, 3.0]
661                                             [1.0]
677                                             [3.0]
1230                                [39.0, 2.0, 19.0]
1237              [3.0, 3.0, 5.0, 2.0, 3.0, 4.0, 8.0]
1361              [5.0, 1.0, 3.0, 1.0, 2.0, 2.0, 5.0]
1404                                            [3.0]
1407                             [9.0, 9.0, 3.0, 3.0]
1415                                       [1.0, 2.0]
1607    [24.0, 6.0, 23.0, 4.0, 5.0, 21.0, 14.0, 26.0]
1609              [2.0, 1.0, 9.0, 4.0, 6.0, 2.0, 5.0]
1621                   [6.0, 3.0, 5.0, 1.0, 7.0, 2.0]
1651                                            [2.0]
1652              [2.0

In [34]:
arrays = [wasserstein_distance(i,j) for i in x for j in x]
arr = np.array(arrays).reshape((len(x),len(x)))
arr

array([[ 0.        ,  0.66666667,  0.25      ,  0.875     ,  3.14285714,
         1.33333333,  1.        ,  1.        , 18.        ,  2.        ,
         0.85714286,  1.        ,  4.        ,  0.5       , 13.375     ,
         2.14285714,  2.        ,  1.        ,  0.42857143,  2.5       ,
         0.5       ,  2.14285714,  1.5       ,  1.375     ,  1.25      ,
         1.        ,  1.        ,  0.25      ],
       [ 0.66666667,  0.        ,  0.41666667,  0.875     ,  3.14285714,
         1.66666667,  1.        ,  1.66666667, 18.        ,  2.        ,
         0.71428571,  1.66666667,  4.        ,  0.5       , 13.375     ,
         2.14285714,  2.        ,  1.        ,  0.42857143,  2.5       ,
         0.5       ,  2.04761905,  1.83333333,  1.375     ,  0.75      ,
         1.        ,  1.        ,  0.91666667],
       [ 0.25      ,  0.41666667,  0.        ,  0.625     ,  3.39285714,
         1.58333333,  0.75      ,  1.25      , 18.25      ,  2.25      ,
         0.96428571,  1.25  

Filling empty bins with 0

In [35]:
flights_freq = flights.groupby(['tagID','date']).size().unstack(fill_value=0)
flights_freq

date,06/21,06/22,06/23,06/24,06/25,06/26,06/27,06/28
tagID,,,,,,,,
197,3,0,0,1,0,0,0,0
508,1,0,0,1,2,2,5,1
515,3,2,1,0,0,0,1,0
516,1,1,2,1,1,1,1,1
517,4,8,7,2,2,7,6,0
638,4,4,4,3,0,2,3,0
661,0,1,0,0,0,0,0,0
677,3,0,0,0,0,0,0,0
1230,39,2,19,0,0,0,0,0


In [36]:
x = flights_freq.apply(lambda x: [x.mean(), x.std(), stats.variation(x), stats.skew(x)], axis=1)
x

tagID
197                 [0.5, 1.0690449676496976, 2.0, 1.875]
508     [1.5, 1.6035674514745464, 1.0, 1.3333333333333...
515     [0.875, 1.1259916264596033, 1.203735681882337,...
516     [1.125, 0.3535533905932738, 0.2939723678960656...
517     [4.5, 2.9277002188455996, 0.6085806194501846, ...
638     [2.5, 1.6903085094570331, 0.6324555320336759, ...
661     [0.125, 0.3535533905932738, 2.6457513110645907...
677     [0.375, 1.0606601717798212, 2.6457513110645907...
1230    [7.5, 14.322808982279179, 1.786368134263235, 1...
1237    [3.5, 2.32992949004287, 0.6226998490772392, 0....
1361    [2.375, 1.8468119248354136, 0.7273828926886976...
1404    [0.375, 1.0606601717798212, 2.6457513110645907...
1407    [3.0, 3.927922024247863, 1.2247448713915892, 0...
1415    [0.375, 0.7440238091428449, 1.8559214542766738...
1607    [15.375, 9.28804915700047, 0.5650845160489477,...
1609    [3.625, 2.9730936268953445, 0.7671929469409307...
1621    [3.0, 2.7255405754769875, 0.8498365855987974, ...
1651    

In [37]:
array = np.array(x.tolist())
squareform(pdist(array))

array([[ 0.        ,  1.60596297,  1.32023758,  1.99182454,  5.09982863,
         3.56413593,  1.10626826,  0.76614069, 14.99305811,  3.76362954,
         2.79808605,  0.76614069,  4.01803783,  0.48842014, 17.17865664,
         4.0779756 ,  3.58997495,  0.87451444,  2.2105665 ,  3.0260306 ,
         2.23969147,  4.84976968,  0.96217355,  3.76412973,  1.62590944,
         1.10626826,  1.10626826,  1.76817803],
       [ 1.60596297,  0.        ,  0.92437193,  1.75352175,  3.67282061,
         2.26545055,  2.65233122,  2.26761417, 14.08761855,  2.28869099,
         1.30990426,  2.26761417,  2.82316395,  1.67044586, 15.93897929,
         2.65282685,  2.15841755,  2.43881989,  1.2320962 ,  2.05786951,
         0.99477558,  3.84440883,  0.98521029,  2.28965945,  1.14941161,
         2.65233122,  2.65233122,  0.95583461],
       [ 1.32023758,  0.92437193,  0.        ,  1.83793002,  4.25459813,
         2.38928164,  2.26488996,  2.05544008, 14.79412035,  2.96237635,
         1.79188673,  2.0554

In [38]:
x = flights_freq.apply(lambda x: x.tolist(), axis=1)
x

tagID
197          [3, 0, 0, 1, 0, 0, 0, 0]
508          [1, 0, 0, 1, 2, 2, 5, 1]
515          [3, 2, 1, 0, 0, 0, 1, 0]
516          [1, 1, 2, 1, 1, 1, 1, 1]
517          [4, 8, 7, 2, 2, 7, 6, 0]
638          [4, 4, 4, 3, 0, 2, 3, 0]
661          [0, 1, 0, 0, 0, 0, 0, 0]
677          [3, 0, 0, 0, 0, 0, 0, 0]
1230       [39, 2, 19, 0, 0, 0, 0, 0]
1237         [3, 3, 5, 2, 3, 4, 8, 0]
1361         [5, 1, 3, 1, 2, 2, 5, 0]
1404         [3, 0, 0, 0, 0, 0, 0, 0]
1407         [9, 9, 3, 3, 0, 0, 0, 0]
1415         [1, 2, 0, 0, 0, 0, 0, 0]
1607    [24, 6, 23, 4, 5, 21, 14, 26]
1609         [2, 1, 9, 4, 6, 2, 5, 0]
1621         [6, 3, 5, 1, 7, 2, 0, 0]
1651         [2, 0, 0, 0, 0, 0, 0, 0]
1652         [2, 1, 1, 1, 2, 1, 3, 0]
1698        [10, 2, 2, 4, 0, 0, 0, 0]
1722         [1, 3, 0, 2, 2, 1, 4, 0]
1780        [1, 2, 2, 1, 0, 15, 3, 4]
1786         [4, 3, 0, 0, 0, 0, 0, 0]
1797         [5, 3, 1, 1, 6, 3, 7, 1]
1900         [1, 1, 2, 7, 0, 0, 0, 0]
1979         [1, 0, 0, 0, 0, 0, 0, 0]
2258  

Full matrix

In [39]:
arrays = [wasserstein_distance(i,j) for i in x for j in x]
arr = np.array(arrays).reshape((len(x),len(x)))

In [40]:
arr

array([[ 0.   ,  1.   ,  0.375,  0.875,  4.   ,  2.   ,  0.375,  0.125,
         7.   ,  3.   ,  1.875,  0.125,  2.5  ,  0.125, 14.875,  3.125,
         2.5  ,  0.25 ,  0.875,  1.75 ,  1.125,  3.   ,  0.375,  2.875,
         0.875,  0.375,  0.375,  0.625],
       [ 1.   ,  0.   ,  0.625,  0.875,  3.   ,  1.25 ,  1.375,  1.125,
         6.75 ,  2.   ,  0.875,  1.125,  2.   ,  1.125, 13.875,  2.125,
         1.5  ,  1.25 ,  0.375,  1.25 ,  0.375,  2.   ,  0.875,  1.875,
         0.625,  1.375,  1.375,  0.625],
       [ 0.375,  0.625,  0.   ,  0.75 ,  3.625,  1.625,  0.75 ,  0.5  ,
         6.875,  2.625,  1.5  ,  0.5  ,  2.125,  0.5  , 14.5  ,  2.75 ,
         2.125,  0.625,  0.5  ,  1.375,  0.75 ,  2.625,  0.5  ,  2.5  ,
         0.5  ,  0.75 ,  0.75 ,  0.25 ],
       [ 0.875,  0.875,  0.75 ,  0.   ,  3.625,  1.875,  1.   ,  1.   ,
         7.625,  2.625,  1.5  ,  1.   ,  2.875,  0.75 , 14.25 ,  2.75 ,
         2.375,  0.875,  0.5  ,  2.125,  1.   ,  2.625,  1.25 ,  2.25 ,
         1.25

In [41]:
squareform(arr)

array([ 1.   ,  0.375,  0.875,  4.   ,  2.   ,  0.375,  0.125,  7.   ,
        3.   ,  1.875,  0.125,  2.5  ,  0.125, 14.875,  3.125,  2.5  ,
        0.25 ,  0.875,  1.75 ,  1.125,  3.   ,  0.375,  2.875,  0.875,
        0.375,  0.375,  0.625,  0.625,  0.875,  3.   ,  1.25 ,  1.375,
        1.125,  6.75 ,  2.   ,  0.875,  1.125,  2.   ,  1.125, 13.875,
        2.125,  1.5  ,  1.25 ,  0.375,  1.25 ,  0.375,  2.   ,  0.875,
        1.875,  0.625,  1.375,  1.375,  0.625,  0.75 ,  3.625,  1.625,
        0.75 ,  0.5  ,  6.875,  2.625,  1.5  ,  0.5  ,  2.125,  0.5  ,
       14.5  ,  2.75 ,  2.125,  0.625,  0.5  ,  1.375,  0.75 ,  2.625,
        0.5  ,  2.5  ,  0.5  ,  0.75 ,  0.75 ,  0.25 ,  3.625,  1.875,
        1.   ,  1.   ,  7.625,  2.625,  1.5  ,  1.   ,  2.875,  0.75 ,
       14.25 ,  2.75 ,  2.375,  0.875,  0.5  ,  2.125,  1.   ,  2.625,
        1.25 ,  2.25 ,  1.25 ,  1.   ,  1.   ,  1.   ,  2.   ,  4.375,
        4.125,  7.75 ,  1.25 ,  2.125,  4.125,  2.25 ,  4.125, 10.875,
      

In [42]:
np.allclose(arr, arr.T)

True

### DURATION SIMILARITY (D2)

#### TIME BETWEEN FLIGHTS

In [43]:
dormant

,location,start,end,age,activity,tagID,duration,day,daydiff
0,Inside,2017-06-21 08:07:56.250,2017-06-21 08:11:24.300,0,inside,197,208.05,21,0
2,Inside,2017-06-21 08:28:03.650,2017-06-21 08:37:01.650,0,inside,197,538.00,21,0
4,Inside,2017-06-21 08:58:19.050,2017-06-21 09:04:15.700,0,inside,197,356.65,21,0
21,Inside,2017-06-24 09:07:51.850,2017-06-24 10:24:33.900,3,inside,197,4602.05,24,0
28,Inside,2017-06-23 09:56:07.000,2017-06-23 12:02:08.550,1,inside,508,7561.55,23,0
...,...,...,...,...,...,...,...,...,...
1812,Inside,2017-06-22 08:03:39.250,2017-06-22 08:23:36.650,0,inside,2487,1197.40,22,0
1815,Inside,2017-06-22 10:58:54.750,2017-06-22 11:11:32.800,1,inside,2487,758.05,22,0
1819,Inside,2017-06-22 15:16:52.900,2017-06-22 16:13:58.850,1,inside,2487,3425.95,22,0
1822,Inside,2017-06-23 09:38:04.300,2017-06-23 09:56:09.200,2,inside,2487,1084.90,23,0


In [44]:
dormant_times = dormant.groupby(['tagID'])['duration'].apply(list)
dormant_times

tagID
197                      [208.05, 538.0, 356.65, 4602.05]
508     [7561.55, 4467.3, 1363.5, 1498.35, 15145.45, 9...
515     [18.95, 8006.4, 455.35, 261.8, 18.65, 14.35, 1...
516     [4928.75, 19353.9, 5195.7, 2286.0, 564.4, 1824...
517     [11129.65, 344.35, 4728.15, 3582.75, 351.2, 32...
638     [750.0, 369.25, 302.4, 376.7, 290.8, 292.35, 2...
661                                                [69.3]
677                               [142.1, 149.45, 1024.9]
930                                             [16828.2]
1230    [426.4, 48.4, 59.65, 42.5, 66.05, 51.7, 42.2, ...
1237    [3034.45, 824.9, 22781.6, 291.0, 18593.65, 217...
1361    [85.35, 136.8, 207.1, 273.0, 196.7, 367.35, 49...
1404                                       [169.0, 458.4]
1407    [1281.45, 12649.55, 51.15, 1090.55, 2116.55, 4...
1415                                     [672.9, 1953.85]
1607    [752.8, 11.4, 286.45, 22.75, 225.4, 12.5, 650....
1609    [474.1, 26413.85, 18117.15, 307.45, 225.4, 137...
1621    

In [45]:
arrays = [wasserstein_distance(i,j) for i in dormant_times for j in dormant_times]
arr = np.array(arrays).reshape((len(dormant_times),len(dormant_times)))
squareform(arr)

array([ 2036.4625    ,  3612.84375   ,  7002.27916667,  1378.91491935,
        1352.23308824,  1356.8875    ,  1068.52083333, 15402.0125    ,
        1406.77      ,  2034.65991379,  1092.48014706,  1112.4875    ,
        1281.4375    ,  1211.2875    ,  1345.52652439,  3673.97142857,
         656.9045    ,  9481.3125    ,  8210.8625    ,   975.34305556,
        3671.10416667,  1922.7625    ,  3169.42916667,  1397.3725    ,
        2749.34305556,  1430.8625    ,  1835.67361111,  2102.31875   ,
        5092.97166667,   789.12564516,  2063.93382353,  3267.295     ,
        2921.72833333, 13492.705     ,  2669.8275    ,  1134.33775862,
        1741.49088235,  3060.945     ,   978.62454545,  2554.2       ,
        2631.41939024,  2679.10107143,  1860.499     ,  7684.05166667,
        6353.885     ,  2870.37277778,  1956.525     ,  1141.82934783,
        1957.11166667,   909.439     ,  2200.99055556,  3015.745     ,
        1045.93055556,  3848.23541667,  2458.32883065,  3594.33933824,
      

In [46]:
np.allclose(arr, arr.T)

True

#### DURATION OF FLIGHTS

In [47]:
active_times = flights.groupby(['tagID'])['duration'].apply(list)
active_times

tagID
197                    [999.35, 1277.4, 5677.9, 19203.15]
508     [6911.35, 2958.65, 3368.35, 5936.05, 1675.5, 1...
515     [30.9, 88.2, 12807.25, 170.4, 3882.5, 61.4, 11...
516     [71.2, 79.4, 118.0, 55.0, 27.25, 38.85, 60.6, ...
517     [77.0, 1031.85, 1421.25, 1829.0, 49.2, 472.15,...
638     [1948.55, 1099.05, 1148.3, 1905.55, 1581.7, 85...
661                                             [8158.35]
677                              [1996.2, 3284.8, 7581.4]
1230    [748.6, 33.8, 44.6, 204.95, 31.5, 27.75, 18.2,...
1237    [767.2, 642.3, 170.35, 2405.9, 5337.4, 1595.85...
1361    [1413.85, 4987.5, 13.0, 571.8, 8565.75, 5358.3...
1404                              [146.75, 889.75, 962.6]
1407    [75.85, 639.75, 470.85, 104.1, 97.65, 10.2, 69...
1415                             [5723.9, 1507.7, 3281.6]
1607    [19.8, 13.75, 26.45, 32.75, 23.3, 20.1, 26.6, ...
1609    [1414.3, 24.1, 738.75, 3233.3, 1332.25, 2515.7...
1621    [854.4, 1013.6, 2159.55, 2792.05, 3506.2, 276....
1651    

In [48]:
arrays = [wasserstein_distance(i,j) for i in active_times for j in active_times]
arr = np.array(arrays).reshape((len(active_times),len(active_times)))
squareform(arr)

array([3.62078750e+03, 3.14412857e+03, 6.72987778e+03, 6.01689028e+03,
       5.49105500e+03, 6.89130000e+03, 4.10659167e+03, 6.62237667e+03,
       4.75179286e+03, 3.40683553e+03, 6.12308333e+03, 5.96965833e+03,
       4.25334167e+03, 6.64247886e+03, 5.66237586e+03, 4.56171458e+03,
       6.64417500e+03, 4.80895682e+03, 6.75605556e+03, 3.92960962e+03,
       6.16673750e+03, 3.97769643e+03, 5.01067593e+03, 6.74125000e+03,
       6.66355000e+03, 1.02039250e+04, 4.47856389e+03, 2.76760298e+03,
       4.46969028e+03, 3.75670278e+03, 3.23086750e+03, 4.31622917e+03,
       1.04323750e+03, 4.36218917e+03, 2.49160536e+03, 1.70391952e+03,
       3.86289583e+03, 3.70947083e+03, 1.07868750e+03, 4.38229136e+03,
       3.40218836e+03, 2.12683125e+03, 4.38398750e+03, 2.49028977e+03,
       4.49586806e+03, 7.36980449e+02, 3.90655000e+03, 1.41688393e+03,
       2.74312361e+03, 4.48106250e+03, 4.40336250e+03, 1.02543375e+04,
       2.06636806e+03, 3.98596667e+03, 3.56754702e+03, 3.55290143e+03,
      

In [49]:
np.allclose(arr, arr.T)

True

### SCHEDULE SIMILARITY (D3)

Bin calculation

In [50]:
active_times

tagID
197                    [999.35, 1277.4, 5677.9, 19203.15]
508     [6911.35, 2958.65, 3368.35, 5936.05, 1675.5, 1...
515     [30.9, 88.2, 12807.25, 170.4, 3882.5, 61.4, 11...
516     [71.2, 79.4, 118.0, 55.0, 27.25, 38.85, 60.6, ...
517     [77.0, 1031.85, 1421.25, 1829.0, 49.2, 472.15,...
638     [1948.55, 1099.05, 1148.3, 1905.55, 1581.7, 85...
661                                             [8158.35]
677                              [1996.2, 3284.8, 7581.4]
1230    [748.6, 33.8, 44.6, 204.95, 31.5, 27.75, 18.2,...
1237    [767.2, 642.3, 170.35, 2405.9, 5337.4, 1595.85...
1361    [1413.85, 4987.5, 13.0, 571.8, 8565.75, 5358.3...
1404                              [146.75, 889.75, 962.6]
1407    [75.85, 639.75, 470.85, 104.1, 97.65, 10.2, 69...
1415                             [5723.9, 1507.7, 3281.6]
1607    [19.8, 13.75, 26.45, 32.75, 23.3, 20.1, 26.6, ...
1609    [1414.3, 24.1, 738.75, 3233.3, 1332.25, 2515.7...
1621    [854.4, 1013.6, 2159.55, 2792.05, 3506.2, 276....
1651    

In [51]:
bins = pd.Series([x for xs in active_times for x in xs]).apply(
    lambda x: 5 if x <= 300 else
    10 if x <= 600 else
    15 if x <= 900 else
    20 if x <= 1200 else
    30 if x <= 1800 else
    60
)

bins

0      20
1      30
2      60
3      60
4      60
       ..
528    60
529    60
530    30
531    60
532     5
Length: 533, dtype: int64

In [52]:
bins.groupby(bins).size()

5     289
10     16
15     36
20     35
30     41
60    116
dtype: int64

Using 5 minute bins

In [53]:
flights_count = flights_count.sort_values(['day','5minute'])
count = flights_count.groupby(['tagID','date','5minute']).size()
idx = pd.MultiIndex.from_product([flights_count.tagID.unique(), flights_count.day.unique(), flights_count['20minute'].unique()])
count = count.reindex(idx).fillna(0)
count
#unstacked = count.unstack(['day','20minute'], fill_value=0)
#unstacked_dict = unstacked.groupby(level=0).apply(lambda df: df.xs(df.name).to_dict()).to_dict()

516  21  8.000000     0.0
         8.333333     0.0
         8.666667     0.0
         9.000000     0.0
         9.333333     0.0
                     ... 
661  28  18.000000    0.0
         18.333333    0.0
         18.666667    0.0
         19.333333    0.0
         19.666667    0.0
Length: 7840, dtype: float64

In [54]:
result = {
    tag: {
        day: g.values.tolist()
        for day, g in tag_group.groupby(level=1)
    }
    for tag, tag_group in count.groupby(level=0)
}
result

{197: {21: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  22: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  23: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  24: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,


In [55]:
def compareScheduleDaytoDay(tag1, tag2, result):
    bee1 = result[tag1]
    bee2 = result[tag2]
    values = []
    for (k,v), (k2,v2) in zip(bee1.items(), bee2.items()):
        cosine = np.dot(v, v2) / (norm(v) * norm(v2))
        values.append(cosine)
    return np.nan_to_num(values)

compareScheduleDaytoDay(1621,1361,result)

C:\Users\kashi\AppData\Local\Temp\ipykernel_2212\371221653.py:6: RuntimeWarning: invalid value encountered in scalar divide
  cosine = np.dot(v, v2) / (norm(v) * norm(v2))


array([0., 0., 0., 0., 0., 0., 0., 0.])

In [56]:
result_mat = {}
for key,val in result.items():
    mat = []
    for day, lst in val.items():
        mat.append(lst)
    npmat = np.array([np.array(xi) for xi in mat])
    result_mat[key] = npmat

result_mat

{197: array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         

In [57]:
def compareMatrixes(tag1,tag2,result):
    bee1 = result[tag1]
    bee2 = result[tag2]
    return cosine_similarity(bee1,bee2)

np.triu(compareMatrixes(1621,1361,result_mat))

array([[0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.]])

### TREND SIMILARITY (D4)

#### FLIGHT FREQUENCY

In [58]:
from sklearn.linear_model import LinearRegression 

In [59]:
flights_nofill

date,06/21,06/22,06/23,06/24,06/25,06/26,06/27,06/28
tagID,,,,,,,,
197,3.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN
508,1.0,NaN,NaN,1.0,2.0,2.0,5.0,1.0
515,3.0,2.0,1.0,NaN,NaN,NaN,1.0,NaN
516,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0
517,4.0,8.0,7.0,2.0,2.0,7.0,6.0,NaN
638,4.0,4.0,4.0,3.0,NaN,2.0,3.0,NaN
661,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
677,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1230,39.0,2.0,19.0,NaN,NaN,NaN,NaN,NaN


In [60]:
reg = flights_nofill.apply(
    lambda x: [(d,v) for d,v in x.items() if pd.notna(v)],
    axis=1
)
reg

tagID
197                          [(06/21, 3.0), (06/24, 1.0)]
508     [(06/21, 1.0), (06/24, 1.0), (06/25, 2.0), (06...
515     [(06/21, 3.0), (06/22, 2.0), (06/23, 1.0), (06...
516     [(06/21, 1.0), (06/22, 1.0), (06/23, 2.0), (06...
517     [(06/21, 4.0), (06/22, 8.0), (06/23, 7.0), (06...
638     [(06/21, 4.0), (06/22, 4.0), (06/23, 4.0), (06...
661                                        [(06/22, 1.0)]
677                                        [(06/21, 3.0)]
1230         [(06/21, 39.0), (06/22, 2.0), (06/23, 19.0)]
1237    [(06/21, 3.0), (06/22, 3.0), (06/23, 5.0), (06...
1361    [(06/21, 5.0), (06/22, 1.0), (06/23, 3.0), (06...
1404                                       [(06/21, 3.0)]
1407    [(06/21, 9.0), (06/22, 9.0), (06/23, 3.0), (06...
1415                         [(06/21, 1.0), (06/22, 2.0)]
1607    [(06/21, 24.0), (06/22, 6.0), (06/23, 23.0), (...
1609    [(06/21, 2.0), (06/22, 1.0), (06/23, 9.0), (06...
1621    [(06/21, 6.0), (06/22, 3.0), (06/23, 5.0), (06...
1651    

In [61]:
days = pd.Series(np.sort(flights['date'].unique()))
day_map = dict(zip(days, days.astype('category').cat.codes))
day_map

{'06/21': 0,
 '06/22': 1,
 '06/23': 2,
 '06/24': 3,
 '06/25': 4,
 '06/26': 5,
 '06/27': 6,
 '06/28': 7}

In [62]:
regs = {}
for index, val in reg.items():
    x = []
    y = []
    for v in val:
        x.append(day_map[v[0]])
        y.append(v[1])
    x = np.array(x).reshape([-1,1])
    linreg = LinearRegression().fit(x, y)
    slope = linreg.coef_
    regs[index] = slope

regs = pd.Series(regs)
regs

197      [-0.6666666666666665]
508       [0.2594594594594595]
515      [-0.2771084337349398]
516     [-0.03571428571428572]
517     [-0.03571428571428576]
638     [-0.28571428571428575]
661                      [0.0]
677                      [0.0]
1230      [-9.999999999999995]
1237      [0.5357142857142857]
1361     [0.03571428571428568]
1404                     [0.0]
1407     [-2.3999999999999995]
1415      [0.9999999999999998]
1607      [0.5833333333333334]
1609     [0.28571428571428564]
1621    [-0.34285714285714297]
1651                     [0.0]
1652     [0.14285714285714282]
1698     [-1.7999999999999998]
1722     [0.18012422360248453]
1780      [0.8150684931506852]
1786     [-0.9999999999999998]
1797    [0.035714285714285705]
1900                     [1.9]
1979                     [0.0]
2258                     [0.0]
2487    [-0.49999999999999983]
dtype: object

In [63]:
arrays = [abs(i-j) for index,i in regs.items() for index2, j in regs.items()]
arr = np.array(arrays).reshape((len(regs),len(regs)))
squareform(arr)

array([9.26126126e-01, 3.89558233e-01, 6.30952381e-01, 6.30952381e-01,
       3.80952381e-01, 6.66666667e-01, 6.66666667e-01, 9.33333333e+00,
       1.20238095e+00, 7.02380952e-01, 6.66666667e-01, 1.73333333e+00,
       1.66666667e+00, 1.25000000e+00, 9.52380952e-01, 3.23809524e-01,
       6.66666667e-01, 8.09523810e-01, 1.13333333e+00, 8.46790890e-01,
       1.48173516e+00, 3.33333333e-01, 7.02380952e-01, 2.56666667e+00,
       6.66666667e-01, 6.66666667e-01, 1.66666667e-01, 5.36567893e-01,
       2.95173745e-01, 2.95173745e-01, 5.45173745e-01, 2.59459459e-01,
       2.59459459e-01, 1.02594595e+01, 2.76254826e-01, 2.23745174e-01,
       2.59459459e-01, 2.65945946e+00, 7.40540541e-01, 3.23873874e-01,
       2.62548263e-02, 6.02316602e-01, 2.59459459e-01, 1.16602317e-01,
       2.05945946e+00, 7.93352359e-02, 5.55609034e-01, 1.25945946e+00,
       2.23745174e-01, 1.64054054e+00, 2.59459459e-01, 2.59459459e-01,
       7.59459459e-01, 2.41394148e-01, 2.41394148e-01, 8.60585198e-03,
      

#### FLIGHT DURATION

In [64]:
dur_nofill = flights.groupby(['tagID','date'])['duration'].agg("mean").unstack()
regDur = dur_nofill.apply(
    lambda x: [(d,v) for d,v in x.items() if pd.notna(v)],
    axis=1
)

In [65]:
regsDur = {}
for index, val in regDur.items():
    x = []
    y = []
    for v in val:
        x.append(day_map[v[0]])
        y.append(v[1])
    x = np.array(x).reshape([-1,1])
    linreg = LinearRegression().fit(x, y)
    slope = linreg.coef_
    regsDur[index] = slope

regsDur = pd.Series(regsDur)
regsDur

197                  [5517.2]
508     [-325.33556756756764]
515      [1447.6722891566262]
516      [-5.449404761904763]
517     [-17.394642857142877]
638      [-89.65527950310559]
661                     [0.0]
677                     [0.0]
1230     [124.22705802968956]
1237             [168.664375]
1361      [403.6204166666664]
1404                    [0.0]
1407      [1050.171111111111]
1415    [-3329.2499999999977]
1607    [-22.366438620041112]
1609     [20.008908730158744]
1621      [555.8883129251701]
1651                    [0.0]
1652      [91.07857142857131]
1698                  [9.942]
1722      [287.0589285714285]
1780     [271.39392123287683]
1786     [1104.2041666666662]
1797       [159.994149659864]
1900    [-11.908928571428572]
1979                    [0.0]
2258                    [0.0]
2487    [-61.173333333333304]
dtype: object

In [66]:
arrays = [abs(i-j) for index,i in regsDur.items() for index2, j in regsDur.items()]
arr = np.array(arrays).reshape((len(regsDur),len(regsDur)))
squareform(arr)

array([5.84253557e+03, 4.06952771e+03, 5.52264940e+03, 5.53459464e+03,
       5.60685528e+03, 5.51720000e+03, 5.51720000e+03, 5.39297294e+03,
       5.34853562e+03, 5.11357958e+03, 5.51720000e+03, 4.46702889e+03,
       8.84645000e+03, 5.53956644e+03, 5.49719109e+03, 4.96131169e+03,
       5.51720000e+03, 5.42612143e+03, 5.50725800e+03, 5.23014107e+03,
       5.24580608e+03, 4.41299583e+03, 5.35720585e+03, 5.52910893e+03,
       5.51720000e+03, 5.51720000e+03, 5.57837333e+03, 1.77300786e+03,
       3.19886163e+02, 3.07940925e+02, 2.35680288e+02, 3.25335568e+02,
       3.25335568e+02, 4.49562626e+02, 4.93999943e+02, 7.28955984e+02,
       3.25335568e+02, 1.37550668e+03, 3.00391443e+03, 3.02969129e+02,
       3.45344476e+02, 8.81223880e+02, 3.25335568e+02, 4.16414139e+02,
       3.35277568e+02, 6.12394496e+02, 5.96729489e+02, 1.42953973e+03,
       4.85329717e+02, 3.13426639e+02, 3.25335568e+02, 3.25335568e+02,
       2.64162234e+02, 1.45312169e+03, 1.46506693e+03, 1.53732757e+03,
      

#### DIFFERENCES

In [67]:
CoeffDif = abs(regsDur - regs)
CoeffDif

197      [5517.866666666667]
508      [325.5950270270271]
515      [1447.949397590361]
516      [5.413690476190477]
517     [17.358928571428592]
638       [89.3695652173913]
661                    [0.0]
677                    [0.0]
1230    [134.22705802968954]
1237    [168.12866071428573]
1361    [403.58470238095214]
1404                   [0.0]
1407    [1052.5711111111111]
1415    [3330.2499999999977]
1607    [22.949771953374444]
1609     [19.72319444444446]
1621     [556.2311700680273]
1651                   [0.0]
1652     [90.93571428571417]
1698                [11.742]
1722      [286.878804347826]
1780    [270.57885273972613]
1786    [1105.2041666666662]
1797    [159.95843537414973]
1900    [13.808928571428572]
1979                   [0.0]
2258                   [0.0]
2487    [60.673333333333304]
dtype: object

#### MULTIVARIATE

In [68]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.multivariate.multivariate_ols import _MultivariateOLS

In [69]:
for i in range(len(regDur)):
    x = []
    y = []
    y2 = []
    for j in range(len(regDur.iloc[i])):     
        x.append(day_map[regDur.iloc[i][j][0]])
        y.append(regDur.iloc[i][j][1])
        y2.append(reg.iloc[i][j][1])
    if len(x) < 2:
        continue
    df = pd.DataFrame({
        'x':x,
        'y1':y,
        'y2':y2
    })
    Y = df[['y1','y2']]
    #X = sm.add_constant(df[['x']])
    X = np.array(df['x']).reshape(-1,1)
    model = _MultivariateOLS(Y, X)
    result = model.fit()
    "print(result.mv_test((('x', [[0, 1]]), None)))
    


SyntaxError: unterminated string literal (detected at line 21) (3945861338.py, line 21)